# Lab MLOps — Versioning de modèles sur le Hub

**Idée :** on n'entraîne jamais *un* modèle, mais plusieurs essais. On va entraîner **3 variantes** du classifieur de sentiment, publier chacune comme une **version** (commit + tag) sur un même dépôt, les **comparer** sur un test commun, puis **promouvoir** la meilleure.

| Version | Ce qui change |
|---|---|
| **v1** baseline | `rotten_tomatoes`, `max_length=128` |
| **v2** preprocessing | `rotten_tomatoes`, `max_length=256` |
| **v3** dataset | sous-ensemble `imdb`, `max_length=256` |

> Entraînements volontairement **courts** (sous-échantillon + 1 epoch) pour enchaîner les 3 variantes. Le but pédagogique est le **workflow de versioning**, pas la performance absolue.

**Environnement :** Colab GPU T4.


## 0. Installation, GPU, login

In [ ]:
!pip install -q -U transformers datasets evaluate accelerate
import torch, numpy as np
print("GPU:", torch.cuda.is_available())
from huggingface_hub import notebook_login
notebook_login()

## 1. Une fonction d'entraînement réutilisable

On factorise l'entraînement dans une fonction : on lui passe le dataset et le `max_length`, elle renvoie un `Trainer` entraîné. Cela évite de dupliquer le code pour chaque variante.

In [ ]:
from transformers import (AutoTokenizer, DataCollatorWithPadding,
    AutoModelForSequenceClassification, TrainingArguments, Trainer)
import evaluate

CHECKPOINT = "distilbert-base-uncased"
REPO = "sentiment-versioning-demo"   # nom du dépôt (sera <user>/sentiment-versioning-demo)
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.argmax(preds, axis=1)
    return accuracy.compute(predictions=preds, references=labels)

def train_variant(train_ds, eval_ds, max_length, run_name):
    tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
    def prep(b): return tokenizer(b["text"], truncation=True, max_length=max_length)
    tr = train_ds.map(prep, batched=True)
    ev = eval_ds.map(prep, batched=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        CHECKPOINT, num_labels=2, id2label=id2label, label2id=label2id)
    args = TrainingArguments(
        output_dir=REPO,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=1,
        eval_strategy="epoch",
        save_strategy="no",
        report_to="none",
        push_to_hub=True,
        run_name=run_name,
    )
    trainer = Trainer(model=model, args=args,
        train_dataset=tr, eval_dataset=ev,
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics)
    trainer.train()
    return trainer

## 2. Préparer les données (sous-échantillons pour aller vite)

In [ ]:
from datasets import load_dataset

rt = load_dataset("rotten_tomatoes")
rt_train = rt["train"].shuffle(seed=42).select(range(2000))
rt_eval  = rt["validation"]

imdb = load_dataset("imdb")
imdb_train = imdb["train"].shuffle(seed=42).select(range(2000))
# imdb n'a pas de split validation officiel : on prend un morceau du test
imdb_eval  = imdb["test"].shuffle(seed=42).select(range(1000))

print("rotten_tomatoes train:", len(rt_train), "| imdb train:", len(imdb_train))

## 3. Fonction utilitaire pour taguer une version

Après chaque `push_to_hub`, on pose un tag (`v1`, `v2`, `v3`) sur le commit courant de `main`.

In [ ]:
from huggingface_hub import HfApi, whoami
api = HfApi()
REPO_ID = f'{whoami()["name"]}/{REPO}'
print("Dépôt :", REPO_ID)

def tag_current(tag):
    # supprime le tag s'il existe déjà (réexécutions), puis le (re)crée sur main
    try:
        api.delete_tag(REPO_ID, tag=tag)
    except Exception:
        pass
    api.create_tag(REPO_ID, tag=tag, revision="main")
    print(f"Tag {tag} posé sur main.")

## 4. v1 — baseline (rotten_tomatoes, max_length=128)

In [ ]:
t1 = train_variant(rt_train, rt_eval, max_length=128, run_name="v1-baseline")
t1.push_to_hub()
tag_current("v1")

## 5. v2 — preprocessing différent (max_length=256)

In [ ]:
t2 = train_variant(rt_train, rt_eval, max_length=256, run_name="v2-maxlen256")
t2.push_to_hub()
tag_current("v2")

## 6. v3 — dataset différent (imdb, max_length=256)

In [ ]:
t3 = train_variant(imdb_train, imdb_eval, max_length=256, run_name="v3-imdb")
t3.push_to_hub()
tag_current("v3")

## 7. Comparer les versions sur un jeu de test COMMUN

Règle d'or : on compare toutes les versions avec **le même mètre**. On évalue v1/v2/v3 sur le **même** jeu de test (ici le test de `rotten_tomatoes`).

In [ ]:
from transformers import pipeline

common_test = rt["test"]
texts  = common_test["text"]
labels = common_test["label"]

def eval_version(revision):
    clf = pipeline("sentiment-analysis", model=REPO_ID, revision=revision,
                   truncation=True, device=0 if torch.cuda.is_available() else -1)
    preds = clf(texts, batch_size=32)
    y = [label2id[p["label"]] for p in preds]
    return accuracy.compute(predictions=y, references=labels)["accuracy"]

results = {v: eval_version(v) for v in ["v1", "v2", "v3"]}
for v, acc in results.items():
    print(f"{v}: accuracy = {acc:.4f}")

best = max(results, key=results.get)
print("\nMeilleure version :", best)

## 8. Promouvoir la meilleure version

On matérialise la « mise en production » en créant une branche `production` qui pointe sur le meilleur tag. Un déploiement (Space, API) chargera alors `revision="production"` et bénéficiera automatiquement des futures promotions.

In [ ]:
try:
    api.delete_branch(REPO_ID, branch="production")
except Exception:
    pass
api.create_branch(REPO_ID, branch="production", revision=best)
print(f"Branche 'production' créée sur {best}.")
print("Chargement en prod :  AutoModel...from_pretrained(REPO_ID, revision='production')")

## ✅ Débrief

- Vous avez produit **3 versions tracées** (commits + tags) dans un seul dépôt.
- Vous les avez **comparées objectivement** sur un test commun.
- Vous avez **promu** la meilleure via une branche `production` → rollback trivial (repointer la branche).
- Allez voir sur le Hub : onglet **Files and versions** → **History** (les 3 commits) et la liste des **tags**.

**Pont Jour 3 :** MLflow fera la même chose côté auto-géré, en **traçant automatiquement** params et métriques de chaque run, avec un *Model Registry* et des *stages* (Staging/Production).
